# 05.05 Soft-Voting Ensemble (LR + CatBoost + EBM)

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, log_loss, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
REPORTS_DIR = PROJECT_ROOT / "reports" / "phase_05"

split = json.loads((MODELS_DIR / "phase05_split_and_schema.json").read_text(encoding="utf-8"))
df_auth = pd.read_csv(PROJECT_ROOT / split["dataset_authentic"])
df_tr_dev = df_auth.iloc[split["auth_train_indices"]].reset_index(drop=True)
y_dev = df_tr_dev["target"].to_numpy(dtype=int)
folds = split["cv_folds"]

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_t = np.asarray(y_true, dtype=int)
    y_p = np.clip(np.asarray(y_prob, dtype=float), 1e-7, 1.0 - 1e-7)
    y_pred = (y_p >= threshold).astype(int)
    cm = confusion_matrix(y_t, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    bal_acc = 0.5 * (spec + rec)
    bins = np.linspace(0.0, 1.0, 11)
    ece = 0.0
    for i in range(10):
        m = (y_p >= bins[i]) & (y_p < bins[i + 1] if i < 9 else y_p <= bins[i + 1])
        if m.any():
            ece += (m.sum() / len(y_t)) * abs(y_t[m].mean() - y_p[m].mean())
    auc = float(roc_auc_score(y_t, y_p)) if len(np.unique(y_t)) == 2 else float("nan")
    return {
        "accuracy": float(accuracy_score(y_t, y_pred)),
        "precision": float(precision_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "recall": float(rec),
        "specificity": float(spec),
        "balanced_accuracy": float(bal_acc),
        "f1": float(f1_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "macro_f1": float(f1_score(y_t, y_pred, average="macro", zero_division=0)),
        "roc_auc": auc,
        "brier_score": float(brier_score_loss(y_t, y_p)),
        "log_loss": float(log_loss(y_t, y_p, labels=[0, 1])),
        "ece": float(ece),
        "confusion_matrix": cm.tolist(),
    }

p_lr_a = np.load(MODELS_DIR / "lr_oof_probs.npy")
p_cb_a = np.load(MODELS_DIR / "cb_oof_probs.npy")
p_ebm_a = np.load(MODELS_DIR / "ebm_oof_probs.npy")
assert len(p_lr_a) == len(p_cb_a) == len(p_ebm_a) == len(y_dev)

p_ens_a = (p_lr_a + p_cb_a + p_ebm_a) / 3.0
overall_a = compute_binary_metrics(y_dev, p_ens_a)
folds_a = pd.DataFrame([
    {**compute_binary_metrics(y_dev[f["val_indices"]], p_ens_a[f["val_indices"]]), "fold": f["fold"]}
    for f in folds
])

p_lr_b = np.load(MODELS_DIR / "lr_trackB_oof_probs.npy")
p_cb_b = np.load(MODELS_DIR / "cb_trackB_oof_probs.npy")
p_ebm_b = np.load(MODELS_DIR / "ebm_trackB_oof_probs.npy")
p_ens_b = (p_lr_b + p_cb_b + p_ebm_b) / 3.0
overall_b = compute_binary_metrics(y_dev, p_ens_b)

np.save(MODELS_DIR / "ensemble_oof_probs.npy", p_ens_a)
np.save(MODELS_DIR / "ensemble_trackB_oof_probs.npy", p_ens_b)

payload = {
    "model_name": "Soft-Voting Ensemble (LR + CatBoost + EBM)",
    "weights": {"lr": 1/3, "catboost": 1/3, "ebm": 1/3},
    "cv_protocol": "10-Fold StratifiedGroupKFold (Authentic Patients)",
    "n_train_dev_patients": int(len(y_dev)),
    "track_a_oof": overall_a,
    "track_a_fold_mean": folds_a.drop(columns=["fold", "confusion_matrix"]).mean().to_dict(),
    "track_a_fold_std": folds_a.drop(columns=["fold", "confusion_matrix"]).std().to_dict(),
    "track_b_oof": overall_b,
}
(MODELS_DIR / "ensemble_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Soft-Voting Ensemble Track A OOF:", json.dumps(overall_a, indent=2))
print("Soft-Voting Ensemble Track B OOF:", json.dumps(overall_b, indent=2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ConfusionMatrixDisplay.from_predictions(y_dev, (p_ens_a >= 0.5).astype(int), display_labels=["INELIGIBLE", "ELIGIBLE"], ax=axes[0], cmap="YlGn")
axes[0].set_title("Ensemble — Track A OOF CM")
RocCurveDisplay.from_predictions(y_dev, p_ens_a, ax=axes[1]); axes[1].set_title("Ensemble — Track A ROC")
CalibrationDisplay.from_predictions(y_dev, p_ens_a, n_bins=8, ax=axes[2]); axes[2].set_title("Ensemble — Track A Calibration")
plt.tight_layout(); plt.savefig(REPORTS_DIR / "05_ensemble_oof_diagnostics.png", dpi=200); plt.close(fig)
print("Saved Soft-Voting Ensemble artifacts and plots.")


Soft-Voting Ensemble Track A OOF: {
  "accuracy": 0.9263157894736842,
  "precision": 0.9206349206349206,
  "recall": 0.9666666666666667,
  "specificity": 0.8571428571428571,
  "balanced_accuracy": 0.9119047619047619,
  "f1": 0.943089430894309,
  "macro_f1": 0.9193059094770053,
  "roc_auc": 0.9490476190476191,
  "brier_score": 0.057935183571892084,
  "log_loss": 0.2204057111264648,
  "ece": 0.07171551371425705,
  "confusion_matrix": [
    [
      30,
      5
    ],
    [
      2,
      58
    ]
  ]
}
Soft-Voting Ensemble Track B OOF: {
  "accuracy": 0.9157894736842105,
  "precision": 0.9193548387096774,
  "recall": 0.95,
  "specificity": 0.8571428571428571,
  "balanced_accuracy": 0.9035714285714285,
  "f1": 0.9344262295081968,
  "macro_f1": 0.9083895853423336,
  "roc_auc": 0.9338095238095239,
  "brier_score": 0.06117477281003021,
  "log_loss": 0.27402009336038374,
  "ece": 0.07529606526191356,
  "confusion_matrix": [
    [
      30,
      5
    ],
    [
      3,
      57
    ]
  ]
}


Saved Soft-Voting Ensemble artifacts and plots.
